# TradingAgents Role LoRA — Optional Phase D

This Colab/Kaggle notebook fine-tunes **one role at a time** from reviewed conversational JSONL. It does not connect to Alpaca or any trading path.

**Operator gate:** run this only after Phases B/C and the persona evaluator show a persistent role-specific weakness. The recommended target is 200–500 human-reviewed examples with a held-out split.

Current API references: [Unsloth Qwen3 guide](https://unsloth.ai/docs/models/tutorials/qwen3-how-to-run-and-fine-tune), [Unsloth GGUF export](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf), and [TRL conversational datasets](https://huggingface.co/docs/trl/en/sft_trainer).

## 1. Runtime and dependencies

Select a GPU runtime first. Package APIs change quickly; if Colab asks for a runtime restart after installation, restart and continue from the import cell.

In [ ]:
%pip install -q --upgrade unsloth trl datasets accelerate bitsandbytes

In [ ]:
from pathlib import Path
import json
import os

import torch
from datasets import load_dataset

assert torch.cuda.is_available(), "Select a GPU runtime before continuing."
print(torch.cuda.get_device_name(0))

## 2. Configuration

`BASE_MODEL` is deliberately a variable. For the Strategy Researcher, the default matches the 4B Qwen3 tier. For CIO/Risk, replace it with the Unsloth/Hugging Face equivalent of Claude's chosen deep model only if that model fits the free GPU.

In [ ]:
ROLE_NAME = "strategy_researcher"  # strategy_researcher | risk_office_guardian | chief_investment_officer
BASE_MODEL = "unsloth/Qwen3-4B-unsloth-bnb-4bit"
DATASET_PATH = "/content/strategy_researcher_gold.jsonl"
OUTPUT_DIR = f"/content/{ROLE_NAME}_lora"
GGUF_DIR = f"/content/{ROLE_NAME}_gguf"

MAX_SEQ_LENGTH = 4096
TEST_FRACTION = 0.15
SEED = 3407
LORA_RANK = 16
EPOCHS = 2

RUN_TRAINING = False  # Set True only after dataset validation and operator approval.
RUN_EXPORT = False    # Set True only after held-out evaluation passes.

assert ROLE_NAME in {"strategy_researcher", "risk_office_guardian", "chief_investment_officer"}

Upload the reviewed gold JSONL if it is not already present. Generated skeletons with blank assistant messages are rejected.

In [ ]:
if not Path(DATASET_PATH).is_file():
    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            DATASET_PATH = "/content/" + next(iter(uploaded))
    except ImportError:
        pass

assert Path(DATASET_PATH).is_file(), f"Dataset not found: {DATASET_PATH}"
raw = load_dataset("json", data_files=DATASET_PATH, split="train")
print(raw)

## 3. Validate and split before training

Validation enforces one role, three-message conversations, non-empty gold answers, and no pending skeletons. The held-out rows are split before the trainer is created.

In [ ]:
def validate_example(example, index):
    messages = example.get("messages")
    assert isinstance(messages, list) and len(messages) == 3, f"row {index}: expected 3 messages"
    assert [m.get("role") for m in messages] == ["system", "user", "assistant"], f"row {index}: bad roles"
    assert all(isinstance(m.get("content"), str) for m in messages), f"row {index}: non-string content"
    assert messages[0]["content"].strip(), f"row {index}: blank system"
    assert messages[1]["content"].strip(), f"row {index}: blank user"
    assert messages[2]["content"].strip(), f"row {index}: blank gold assistant"
    metadata = example.get("metadata") or {}
    assert metadata.get("role") == ROLE_NAME, f"row {index}: role mismatch"
    assert metadata.get("gold_status") != "pending_human_review", f"row {index}: unreviewed skeleton"

for i, row in enumerate(raw):
    validate_example(row, i)

assert len(raw) >= 20, "Need at least 20 reviewed examples for a meaningful held-out split; target 200–500."
print(f"Validated {len(raw)} reviewed {ROLE_NAME} examples.")

In [ ]:
def to_prompt_completion(example):
    messages = example["messages"]
    return {
        "prompt": messages[:2],
        "completion": [messages[2]],
    }

dataset = raw.map(to_prompt_completion, remove_columns=raw.column_names)
split = dataset.train_test_split(test_size=TEST_FRACTION, seed=SEED, shuffle=True)
train_dataset = split["train"]
eval_dataset = split["test"]
print(f"train={len(train_dataset)} held_out={len(eval_dataset)}")

## 4. Load the quantized base and attach LoRA

The adapter targets the standard attention and MLP projections. Full fine-tuning is intentionally disabled.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_RANK * 2,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)

## 5. Train with completion-only loss

TRL accepts conversational prompt/completion datasets. `completion_only_loss=True` prevents the adapter from learning to reproduce the saved inputs.

In [ ]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    optim="adamw_8bit",
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    completion_only_loss=True,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
)

if RUN_TRAINING:
    train_result = trainer.train()
    print(train_result)
    print(trainer.evaluate())
else:
    print("Training is armed but disabled. Review the split, then set RUN_TRAINING=True.")

## 6. Inspect held-out behavior

Loss alone is insufficient. Review held-out generations for exact facts, required headings, blockers, and authority boundaries before export.

In [ ]:
if RUN_TRAINING:
    FastLanguageModel.for_inference(model)
    sample = eval_dataset[0]
    rendered = tokenizer.apply_chat_template(
        sample["prompt"],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(rendered, return_tensors="pt").to("cuda")
    output = model.generate(
        **inputs,
        max_new_tokens=700,
        do_sample=False,
        use_cache=True,
    )
    generated = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print("MODEL OUTPUT:\n", generated)
    print("\nGOLD OUTPUT:\n", sample["completion"][0]["content"])
else:
    print("Run training first.")

## 7. Export Q4_K_M GGUF

Export only after held-out review. Unsloth documents `q4_k_m` as a supported/recommended GGUF method. Keep the same Qwen chat template when registering with Ollama.

In [ ]:
if RUN_TRAINING and RUN_EXPORT:
    model.save_pretrained_gguf(
        GGUF_DIR,
        tokenizer,
        quantization_method="q4_k_m",
    )
    ggufs = list(Path(GGUF_DIR).glob("*.gguf"))
    assert ggufs, "GGUF export completed without a .gguf file"
    print("Exported:", [str(path) for path in ggufs])
else:
    print("Export disabled. Set RUN_EXPORT=True only after held-out evaluation passes.")

In [ ]:
if RUN_TRAINING and RUN_EXPORT:
    import shutil
    archive = shutil.make_archive(GGUF_DIR, "zip", GGUF_DIR)
    print(archive)
    try:
        from google.colab import files
        files.download(archive)
    except ImportError:
        pass

## 8. Local registration and A/B gate

After downloading, create a new Modelfile that points to the GGUF and preserves the role persona/system prompt. Register under a **new** name such as `ta-quant-ft:latest`, `ta-risk-ft:latest`, or `ta-decider-ft:latest`; do not overwrite the persona-only model.

```text
FROM ./strategy_researcher.Q4_K_M.gguf
PARAMETER temperature 0.2
PARAMETER num_ctx 4096
PARAMETER num_predict 512
SYSTEM """Paste the matching reviewed role persona here."""
```

Then run the relevant `scripts/eval_personas.py --model ...` cases head-to-head and manually score held-out examples. Keep the fine-tune only if it improves the skill-matrix success metric without weakening blockers, caps, paper-only language, or strategy status.